In [3]:
import pandas as pd
import numpy as np
import datetime
import os
from sqlalchemy import create_engine

# 1. Генерация датасета CDR
np.random.seed(42)
n_records = 10000

subscribers = [f"SUB_{i:05d}" for i in range(1, 501)]
call_types = ['VOICE', 'SMS', 'DATA']
rates = ['STANDARD', 'ROAMING', 'PREMIUM']

start_date = datetime.datetime(2026, 9, 1)

data = {
    'cdr_id': [f"CDR_{i:08d}" for i in range(1, n_records + 1)],
    'subscriber_id': np.random.choice(subscribers, n_records),
    'call_type': np.random.choice(call_types, n_records, p=[0.5, 0.2, 0.3]),
    'timestamp': [start_date + datetime.timedelta(seconds=int(x)) for x in np.random.randint(0, 30*24*3600, n_records)],
    'duration_sec': np.random.exponential(scale=120, size=n_records).astype(int),
    'bytes_used': np.random.exponential(scale=5000000, size=n_records).astype(int),
    'rate_plan': np.random.choice(rates, n_records, p=[0.8, 0.15, 0.05]),
    'charged_amount': np.round(np.random.uniform(0.0, 15.0, n_records), 2)
}

df_cdr = pd.DataFrame(data)

# Корректировка нулевых значений для SMS и DATA
df_cdr.loc[df_cdr['call_type'] == 'SMS', 'duration_sec'] = 0
df_cdr.loc[df_cdr['call_type'] == 'VOICE', 'bytes_used'] = 0

print(f"Сгенерировано {len(df_cdr)} записей CDR.")
display(df_cdr.head())

# 2. Оптимизация типов данных (Pandas -> Parquet)
df_cdr['call_type'] = df_cdr['call_type'].astype('category')
df_cdr['rate_plan'] = df_cdr['rate_plan'].astype('category')
df_cdr['call_date'] = df_cdr['timestamp'].dt.date

# Сохранение в Parquet локально
os.makedirs('../data/processed', exist_ok=True)
df_cdr.to_parquet('../data/processed/raw_cdr.parquet', partition_cols=['call_date'])
print("Saved to Parquet with partitioning by call_date!")

# 3. Загрузка сырых данных в PostgreSQL (staging слой)
db_user = os.getenv('DB_USER', 'analytics_user')
db_pass = os.getenv('DB_PASSWORD', 'analytics_password')
db_host = os.getenv('DB_HOST', 'postgres_dwh')
db_name = os.getenv('DB_NAME', 'telecom_dwh')

engine = create_engine(f'postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:5432/{db_name}')

df_cdr.to_sql('stg_cdr_raw', engine, if_exists='replace', index=False)
print("Успешно загружено в таблицу stg_cdr_raw в PostgreSQL DWH!")

Сгенерировано 10000 записей CDR.


,cdr_id,subscriber_id,call_type,timestamp,duration_sec,bytes_used,rate_plan,charged_amount
0,CDR_00000001,SUB_00103,VOICE,2026-09-27 05:20:34,243,0,STANDARD,2.09
1,CDR_00000002,SUB_00436,VOICE,2026-09-05 02:05:26,101,0,ROAMING,11.07
2,CDR_00000003,SUB_00349,VOICE,2026-09-30 10:01:10,65,0,ROAMING,13.51
3,CDR_00000004,SUB_00271,SMS,2026-09-30 01:47:51,0,1313593,STANDARD,9.85
4,CDR_00000005,SUB_00107,VOICE,2026-09-10 14:43:43,353,0,ROAMING,12.69


Saved to Parquet with partitioning by call_date!
Успешно загружено в таблицу stg_cdr_raw в PostgreSQL DWH!


In [2]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Чтение параметров подключения из переменных окружения
db_user = os.getenv('DB_USER', 'analytics_user')
db_pass = os.getenv('DB_PASSWORD', 'analytics_password')
db_host = os.getenv('DB_HOST', 'postgres_dwh')
db_name = os.getenv('DB_NAME', 'telecom_dwh')

# Явно указываем драйвер +psycopg2
engine = create_engine(f'postgresql+psycopg2://{db_user}:{db_pass}@{db_host}:5432/{db_name}')

# Проверка выполнения SQL-запроса
with engine.connect() as conn:
    result = conn.execute(text("SELECT version();")).fetchone()
    print("✅ Успешное подключение к PostgreSQL DWH!")
    print(f"Информация о СУБД: {result[0]}")

✅ Успешное подключение к PostgreSQL DWH!
Информация о СУБД: PostgreSQL 16.15 on x86_64-pc-linux-musl, compiled by gcc (Alpine 15.2.0) 15.2.0, 64-bit
